[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuelcosta/generative-networks/blob/main/pretraining/lejepa.ipynb)

This notebook is part of the [Generative Networks course](https://www.nahuelcosta.com/teaching/generative-networks).

# Latent-Euclidean JEPA and SIGReg

We train one shared encoder on multiple views with two complementary forces: predictive agreement between related views and explicit matching of each view's batch embeddings to an isotropic Gaussian.

The model in this notebook is deliberately small. It demonstrates a pretraining objective and representation-transfer workflow; it is **not** a foundation model. Foundation models additionally require broad data, much larger capacity, and extensive compute.


The current LeJEPA paper describes the method as **Latent-Euclidean JEPA**. Its theoretical and empirical conclusions are research claims from that work, not universal guarantees for every dataset. Unlike I-JEPA, this implementation has no target teacher, EMA update, stop-gradient, negative pairs, or predictor used to prevent collapse.

For context, I-JEPA encodes the complete image in its target branch and selects target blocks from the target encoder's **output**. We do not implement I-JEPA here; this notebook focuses exclusively on LeJEPA.


## 0. Dependencies


In [ ]:
from importlib import util
import subprocess
import sys

REQUIRED = {"torch": "torch", "torchvision": "torchvision", "matplotlib": "matplotlib", "numpy": "numpy"}
missing = [package for module, package in REQUIRED.items() if util.find_spec(module) is None]
if missing:
    print("Installing missing course dependencies:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Course dependencies are already installed.")


## 1. Imports and Reproducibility


In [ ]:
import copy
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fast_dev_run = os.environ.get("FAST_DEV_RUN", "0") == "1"
print(f"Using device: {device}; FAST_DEV_RUN={fast_dev_run}")


## 2. Configuration


In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "pretraining" else Path.cwd()
output_dir = project_root / "pretraining" / "outputs" / "lejepa"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / "lejepa_encoder.pt"
batch_size = 16 if fast_dev_run else 128
epochs = 1 if fast_dev_run else 10
max_steps = 2 if fast_dev_run else None
learning_rate = 1e-3
lambda_sigreg = 0.05
number_of_slices = 32 if fast_dev_run else 512
global_views, local_views = 2, 4
print("Artifacts:", output_dir)
print(f"Views: {global_views} global + {local_views} local; SIGReg slices={number_of_slices}")


The paper recommends two global and eight local views, at least 128 samples per batch, and 1,024 slices as general starting points. Four local views and 512 slices are a deliberate course-compute reduction. The loss still follows the published construction, with 17 Epps–Pulley integration points over `[-5, 5]` and `λ=0.05`.


## 3. CIFAR-10 and Multi-Crop Views


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
data_root = project_root / "data"

evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
raw_train = datasets.CIFAR10(data_root, train=True, download=True)
probe_source = datasets.CIFAR10(data_root, train=True, transform=evaluation_transform)
test_source = datasets.CIFAR10(data_root, train=False, download=True, transform=evaluation_transform)

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(raw_train), generator=generator).tolist()
pretrain_count = 256 if fast_dev_run else 20_000
probe_count = 256 if fast_dev_run else 10_000
test_count = 256 if fast_dev_run else len(test_source)
pretrain_indices = permutation[:pretrain_count]
probe_indices = permutation[:probe_count]

probe_train = Subset(probe_source, probe_indices)
probe_test = Subset(test_source, range(test_count))
loader_kwargs = {"num_workers": 0 if fast_dev_run else 2, "pin_memory": device.type == "cuda"}
probe_train_loader = DataLoader(probe_train, batch_size=256, shuffle=False, **loader_kwargs)
probe_test_loader = DataLoader(probe_test, batch_size=256, shuffle=False, **loader_kwargs)

print(f"Unlabeled pretraining images: {pretrain_count:,}")
print(f"Labeled probe images: {probe_count:,}; test images: {test_count:,}")


In [ ]:
global_transform = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.5, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.1), transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
local_transform = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.2, 0.5)), transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2), transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

class MultiCropViews(Dataset):
    def __init__(self, source, indices):
        self.source, self.indices = source, indices
    def __len__(self): return len(self.indices)
    def __getitem__(self, index):
        image, _ = self.source[self.indices[index]]
        return [global_transform(image) for _ in range(global_views)] + [local_transform(image) for _ in range(local_views)]

pretrain_loader = DataLoader(
    MultiCropViews(raw_train, pretrain_indices), batch_size=batch_size, shuffle=True,
    drop_last=True, generator=torch.Generator().manual_seed(SEED), **loader_kwargs,
)
views = next(iter(pretrain_loader))
assert len(views) == global_views + local_views and all(view.shape[-2:] == (32, 32) for view in views)
shown = torch.cat([view[:4] for view in views])
shown = shown * torch.tensor(CIFAR_STD).view(1, 3, 1, 1) + torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)
plt.figure(figsize=(10, 4)); plt.imshow(make_grid(shown.clamp(0, 1), nrow=4).permute(1, 2, 0)); plt.axis("off")
plt.title("Two global rows followed by four local rows")


## 4. One Shared Encoder and Loss Projection


In [ ]:
class ConvEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
    def forward(self, images):
        return self.network(images).flatten(1)


In [ ]:
class LeJEPAEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = ConvEncoder()
        self.projector = nn.Sequential(nn.Linear(256, 256), nn.GELU(), nn.Linear(256, 128))
    def forward(self, images):
        features = self.backbone(images)
        return features, self.projector(features)

model = LeJEPAEncoder().to(device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")


## 5. Sketched Isotropic Gaussian Regularization

SIGReg samples unit directions, projects the batch onto each direction, and compares the empirical characteristic function with that of `N(0,1)`. This tests more than zero mean and identity covariance: distributions with matching first two moments can still have different characteristic functions.


In [ ]:
def sigreg(embeddings, step, slices=number_of_slices):
    embeddings = embeddings.float()
    generator = torch.Generator(device=embeddings.device).manual_seed(SEED + int(step))
    directions = torch.randn(embeddings.size(1), slices, generator=generator, device=embeddings.device)
    directions = F.normalize(directions, dim=0)
    points = torch.linspace(-5, 5, 17, device=embeddings.device)
    target_cf = torch.exp(-0.5 * points.square())
    projected = (embeddings @ directions).unsqueeze(-1) * points
    empirical_cf = torch.exp(1j * projected).mean(0)
    error = (empirical_cf - target_cf).abs().square() * target_cf
    statistic = torch.trapezoid(error, points, dim=-1) * embeddings.size(0)
    return statistic.mean()

test_generator = torch.Generator().manual_seed(SEED)
gaussian = torch.randn(512, 32, generator=test_generator, device=device)
collapsed = torch.zeros_like(gaussian)
gaussian_score = sigreg(gaussian, step=0, slices=128).item()
collapsed_score = sigreg(collapsed, step=0, slices=128).item()
assert gaussian_score < collapsed_score
print(f"SIGReg Gaussian={gaussian_score:.3f}; collapsed={collapsed_score:.3f}")


## 6. The LeJEPA Objective


In [ ]:
def lejepa_loss(current_model, current_views, step):
    projected = []
    for view in current_views:
        _, embedding = current_model(view)
        projected.append(embedding)
    embeddings = torch.stack(projected)  # [views, batch, dimensions]
    center = embeddings[:global_views].mean(0)
    prediction_loss = (embeddings - center.unsqueeze(0)).square().mean()
    gaussian_loss = torch.stack([sigreg(embedding, step) for embedding in embeddings]).mean()
    total = (1 - lambda_sigreg) * prediction_loss + lambda_sigreg * gaussian_loss
    return total, prediction_loss, gaussian_loss, embeddings

small_views = [view[:8].to(device) for view in views]
total, prediction_loss, gaussian_loss, embeddings = lejepa_loss(model, small_views, step=0)
total.backward()
backbone_gradient = sum(p.grad.abs().sum().item() for p in model.backbone.parameters() if p.grad is not None)
assert math.isfinite(total.item()) and backbone_gradient > 0
assert len([module for module in model.modules() if isinstance(module, LeJEPAEncoder)]) == 1
model.zero_grad(set_to_none=True)
print(f"Initial total={total.item():.3f}; prediction={prediction_loss.item():.3f}; SIGReg={gaussian_loss.item():.3f}")


## 7. Pretrain LeJEPA


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
history, step = [], 0
for epoch in range(epochs):
    model.train(); rows = []
    for views in pretrain_loader:
        views = [view.to(device) for view in views]
        # Complex characteristic functions are evaluated in float32 for stable, portable behavior.
        total, prediction_loss, gaussian_loss, _ = lejepa_loss(model, views, step)
        optimizer.zero_grad(set_to_none=True); total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        rows.append((total.item(), prediction_loss.item(), gaussian_loss.item())); step += 1
        if max_steps is not None and step >= max_steps: break
    epoch_values = np.mean(rows, axis=0).tolist(); history.append(epoch_values)
    print(f"Epoch {epoch + 1:02d}/{epochs:02d} | total={epoch_values[0]:.3f} | prediction={epoch_values[1]:.3f} | SIGReg={epoch_values[2]:.3f}")
    if max_steps is not None and step >= max_steps: break

torch.save({"model": model.state_dict(), "history": history, "lambda": lambda_sigreg}, checkpoint_path)
assert np.isfinite(history).all()
labels = ["total", "prediction", "SIGReg"]
for index, label in enumerate(labels): plt.plot([row[index] for row in history], label=label)
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("LeJEPA loss components"); plt.legend(); plt.grid(alpha=0.3)


## 8. Diagnose the Learned Geometry


In [ ]:
model.eval(); views = next(iter(pretrain_loader))
with torch.no_grad():
    _, embeddings = model(views[0].to(device)); embeddings = embeddings.float().cpu()
centered = embeddings - embeddings.mean(0)
covariance = centered.T @ centered / max(1, embeddings.size(0) - 1)
eigenvalues = torch.linalg.eigvalsh(covariance).clamp_min(1e-12)
probabilities = eigenvalues / eigenvalues.sum()
effective_rank = torch.exp(-(probabilities * probabilities.log()).sum()).item()
direction = F.normalize(torch.randn(embeddings.size(1), generator=torch.Generator().manual_seed(SEED)), dim=0)
projection = embeddings @ direction
print(f"Mean norm={embeddings.mean(0).norm():.3f}; effective covariance rank={effective_rank:.1f}/{embeddings.size(1)}")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(projection.numpy(), bins=30, density=True, alpha=0.8); axes[0].set_title("One random embedding projection")
axes[1].plot(eigenvalues.flip(0).numpy()); axes[1].set_yscale("log"); axes[1].set_title("Covariance spectrum")
for axis in axes: axis.grid(alpha=0.3)
plt.tight_layout()


## 9. Frozen Linear Probe


In [ ]:
@torch.no_grad()
def extract_features(encoder, loader):
    encoder.eval()
    features, labels = [], []
    for images, targets in loader:
        features.append(encoder(images.to(device)).float().cpu())
        labels.append(targets)
    return torch.cat(features), torch.cat(labels)

def fit_linear_probe(encoder, name):
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    train_x, train_y = extract_features(encoder, probe_train_loader)
    test_x, test_y = extract_features(encoder, probe_test_loader)
    head = nn.Linear(train_x.size(1), 10).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=3e-3, weight_decay=1e-4)
    train_loader = DataLoader(
        TensorDataset(train_x, train_y), batch_size=256, shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    epochs = 1 if fast_dev_run else 15
    history = []
    for epoch in range(epochs):
        head.train()
        for features, targets in train_loader:
            logits = head(features.to(device))
            loss = F.cross_entropy(logits, targets.to(device))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        head.eval()
        with torch.no_grad():
            predictions = head(test_x.to(device)).argmax(1).cpu()
        accuracy = (predictions == test_y).float().mean().item()
        history.append(accuracy)
    print(f"{name}: final frozen-probe accuracy={accuracy:.3f}")
    return {"name": name, "accuracy": accuracy, "history": history}


In [ ]:
random_encoder = ConvEncoder()
pretrained_encoder = model.backbone


In [ ]:
probe_results = [
    fit_linear_probe(random_encoder.to(device), "Random frozen encoder"),
    fit_linear_probe(pretrained_encoder.to(device), "Pretrained frozen encoder"),
]
plt.figure(figsize=(6, 4))
plt.bar([row["name"] for row in probe_results], [row["accuracy"] for row in probe_results])
plt.ylabel("CIFAR-10 test accuracy")
plt.xticks(rotation=15, ha="right")
plt.ylim(0, 1)
plt.title("Representation quality through a linear probe")
plt.tight_layout()


## 10. Reload the Checkpoint


In [ ]:
reloaded = LeJEPAEncoder().to(device)
reloaded.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True)["model"])
reloaded.eval(); model.eval(); check = views[0][:4].to(device)
with torch.no_grad():
    assert torch.equal(model.backbone(check), reloaded.backbone(check))
print("Reloaded backbone features match exactly.")


## 11. Things to Try

- Set `lambda_sigreg=0` and monitor covariance rank for collapse.
- Increase from 512 to the paper's recommended 1,024 slices.
- Compare two, six, and ten total views.
- Replace SIGReg with mean/covariance penalties and inspect higher-order projection shapes.

## Troubleshooting

- SIGReg intentionally runs in float32 even when the backbone uses mixed precision.
- Very small batches provide a noisy empirical distribution; the smoke configuration tests plumbing, not geometry quality.
- Linear-probe performance from this short CIFAR run must not be compared directly with ImageNet-scale paper results.

**References:** [LeJEPA v3](https://arxiv.org/abs/2511.08544), [I-JEPA](https://arxiv.org/abs/2301.08243).
